# Sensibilità dell'errore SDE-Net alla threshold MTGFlow

Analisi esclusivamente **post-processing** dei CSV già prodotti. Il notebook non addestra e non modifica MTGFlow o SDE-Net.

Obiettivi:
- asse X: threshold MTGFlow;
- asse Y: MAE/RMSE SDE-Net;
- curve separate per campioni normali e rari;
- numero e percentuale di campioni che cambiano classe;
- esportazione di una threshold candidata e delle soglie effettive per località.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Eseguire il notebook dalla root del repository o da notebooks/.')

MTGFLOW_SEED_DIR = Path(os.environ.get(
    'MTGFLOW_SEED_DIR', ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense' / 'seed_15'
)).resolve()
MTGFLOW_TEST_CSV = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
MTGFLOW_TRAIN_CSV = MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv'
SDE_PREDICTIONS_CSV = Path(os.environ.get(
    'SDE_PREDICTIONS_CSV', ROOT / 'outputs' / 'REPLACE_WITH_SDE_RUN' / 'predictions.csv'
)).resolve()
OUT_DIR = Path(os.environ.get(
    'MTGFLOW_THRESHOLD_OUT_DIR', ROOT / 'outputs' / 'mtgflow_threshold_sensitivity'
)).resolve()

# Paper MTGFlow, Eq. 13: threshold_i(k) = Q3_i + k * IQR_i.
# k=1.5 ricostruisce la threshold originale di ogni località.
IQR_K_VALUES = np.linspace(0.50, 3.00, 51)
PAPER_IQR_K = 1.5
SELECTED_IQR_K = 1.5  # cambiare soltanto dopo l'analisi/validation
TRAIN_QUANTILE_SOURCE = 'full_per_site'  # oppure 'aggregate_export_anchored'
PER_SITE_TRAIN_GLOB = '*/train_scores.csv'
DAYTIME_ONLY = True
DAYTIME_THRESHOLD_WM2 = 10.0
MIN_JOIN_COVERAGE = 0.95
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('MTGFlow test :', MTGFLOW_TEST_CSV)
print('MTGFlow train:', MTGFLOW_TRAIN_CSV)
print('SDE predictions:', SDE_PREDICTIONS_CSV)
print('Output:', OUT_DIR)

## Protocollo

Il CSV MTGFlow contiene una threshold diversa per ogni località, perché ciascun modello viene calibrato sui propri score storici. Lo sweep varia il coefficiente IQR comune `k` dell'equazione 13 del paper:

```text
threshold_i(k) = Q3_i + k × (Q3_i - Q1_i)
```

L'asse X rappresenta `k`: `k=1.5` è la threshold del paper e riproduce la classificazione già salvata; valori inferiori classificano più campioni come rari, valori superiori sono più selettivi. In modalità raccomandata `full_per_site`, Q1 e Q3 sono ricalcolati sui file completi 2005–2018 di ogni località. Il fallback `aggregate_export_anchored` usa l'IQR del CSV storico aggregato disponibile e ancora lo sweep alla threshold salvata: riproduce esattamente il baseline, ma le soglie diverse da `k=1.5` sono un'approssimazione.

In [ ]:
required_paths = [MTGFLOW_TEST_CSV, SDE_PREDICTIONS_CSV]
if TRAIN_QUANTILE_SOURCE == 'aggregate_export_anchored':
    required_paths.append(MTGFLOW_TRAIN_CSV)
missing = [path for path in required_paths if not path.is_file()]
if missing:
    candidates = sorted(ROOT.glob('outputs/**/predictions.csv'))
    print('predictions.csv trovati nel repository:')
    for path in candidates[-20:]:
        print(' -', path)
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))

mtg_header = set(pd.read_csv(MTGFLOW_TEST_CSV, nrows=0).columns)
pred_header = set(pd.read_csv(SDE_PREDICTIONS_CSV, nrows=0).columns)
assert {'location', 'timestamp', 'anomaly_score', 'threshold'} <= mtg_header
if TRAIN_QUANTILE_SOURCE not in {'full_per_site', 'aggregate_export_anchored'}:
    raise ValueError(f'TRAIN_QUANTILE_SOURCE non riconosciuta: {TRAIN_QUANTILE_SOURCE}')
per_site_train_paths = sorted(MTGFLOW_SEED_DIR.glob(PER_SITE_TRAIN_GLOB))
if TRAIN_QUANTILE_SOURCE == 'full_per_site' and not per_site_train_paths:
    raise FileNotFoundError(
        f'Nessun file trovato con {MTGFLOW_SEED_DIR / PER_SITE_TRAIN_GLOB}. '
        'Usare aggregate_export_anchored soltanto se i file completi non sono disponibili.'
    )
if TRAIN_QUANTILE_SOURCE == 'aggregate_export_anchored':
    train_header = set(pd.read_csv(MTGFLOW_TRAIN_CSV, nrows=0).columns)
    assert {'location', 'anomaly_score'} <= train_header
assert {'location', 'timestamp', 'y_true'} <= pred_header
PREDICTION_COLUMN = 'y_pred_mean' if 'y_pred_mean' in pred_header else 'y_pred'
if PREDICTION_COLUMN not in pred_header:
    raise ValueError('predictions.csv deve contenere y_pred_mean oppure y_pred.')
if DAYTIME_ONLY and 'solar_irradiance_poa_target' not in pred_header:
    raise ValueError('DAYTIME_ONLY richiede solar_irradiance_poa_target.')
print('Colonna di previsione:', PREDICTION_COLUMN)

In [ ]:
mtgflow = pd.read_csv(
    MTGFLOW_TEST_CSV,
    usecols=['location', 'timestamp', 'anomaly_score', 'threshold'],
)
train_scores = None
if TRAIN_QUANTILE_SOURCE == 'aggregate_export_anchored':
    train_scores = pd.read_csv(
        MTGFLOW_TRAIN_CSV, usecols=['location', 'anomaly_score']
    )
prediction_columns = ['location', 'timestamp', 'y_true', PREDICTION_COLUMN]
if DAYTIME_ONLY:
    prediction_columns.append('solar_irradiance_poa_target')
predictions = pd.read_csv(SDE_PREDICTIONS_CSV, usecols=prediction_columns)

frames_with_locations = [mtgflow, predictions]
if train_scores is not None:
    frames_with_locations.append(train_scores)
for frame in frames_with_locations:
    frame['location'] = frame['location'].astype(str)
mtgflow['timestamp'] = pd.to_datetime(mtgflow['timestamp'], errors='raise')
predictions['timestamp'] = pd.to_datetime(predictions['timestamp'], errors='raise')
if mtgflow.duplicated(['location', 'timestamp']).any():
    raise ValueError('MTGFlow contiene duplicati location-timestamp.')
if predictions.duplicated(['location', 'timestamp']).any():
    raise ValueError('SDE predictions contiene duplicati location-timestamp.')

joined = predictions.merge(
    mtgflow, on=['location', 'timestamp'], how='inner', validate='one_to_one'
)
coverage = len(joined) / max(len(predictions), 1)
print(f'Predizioni SDE: {len(predictions):,}')
print(f'Righe MTGFlow: {len(mtgflow):,}')
print(f'Righe allineate: {len(joined):,} ({coverage:.2%} delle predizioni)')
if coverage < MIN_JOIN_COVERAGE:
    raise ValueError('Copertura temporale insufficiente: controllare timestamp e score_stride=1.')

numeric = ['y_true', PREDICTION_COLUMN, 'anomaly_score', 'threshold']
joined = joined[np.isfinite(joined[numeric].to_numpy(dtype=float)).all(axis=1)].copy()
if DAYTIME_ONLY:
    joined = joined[joined['solar_irradiance_poa_target'] > DAYTIME_THRESHOLD_WM2].copy()
threshold_variants = mtgflow.groupby('location', observed=True)['threshold'].nunique()
if (threshold_variants != 1).any():
    raise ValueError('Ogni località deve avere una sola threshold MTGFlow salvata.')
saved_thresholds = (
    mtgflow.groupby('location', observed=True)['threshold'].first().rename('saved_threshold')
)
joined['error'] = joined[PREDICTION_COLUMN] - joined['y_true']
joined['abs_error'] = joined['error'].abs()
joined['squared_error'] = joined['error'].square()
print(f'Campioni analizzati: {len(joined):,} ({"daytime" if DAYTIME_ONLY else "tutti"})')
display(joined.head())

## Ricostruzione IQR e asse delle threshold

Per ogni località vengono stimati Q1, Q3 e IQR sugli score di training. Ogni score 2019 viene trasformato nel valore di `k` al quale cambierebbe classe; lo sweep può così essere calcolato ordinando una sola volta i campioni, senza ripetere decine di filtri su milioni di righe.

In [ ]:
def _quantiles(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError('Nessuno score di training finito.')
    q1, q3 = np.quantile(values, [0.25, 0.75])
    return float(q1), float(q3), int(values.size)

def load_full_per_site_quantiles(paths):
    rows = []
    for number, path in enumerate(paths, start=1):
        frame = pd.read_csv(path, usecols=['location', 'anomaly_score'])
        locations = frame['location'].astype(str).unique()
        if len(locations) != 1:
            raise ValueError(f'{path} contiene {len(locations)} località, attesa una.')
        q1, q3, n_scores = _quantiles(frame['anomaly_score'])
        rows.append({'location': locations[0], 'q1': q1, 'q3': q3, 'n_train_scores': n_scores})
        if number % 100 == 0 or number == len(paths):
            print(f'Quantili training: {number}/{len(paths)} file')
    return pd.DataFrame(rows)

if TRAIN_QUANTILE_SOURCE == 'full_per_site':
    quantile_stats = load_full_per_site_quantiles(per_site_train_paths)
    threshold_formula_mode = 'paper_exact_full_training'
else:
    finite_train = train_scores[np.isfinite(train_scores['anomaly_score'])].copy()
    grouped_quantiles = (
        finite_train.groupby('location', observed=True)['anomaly_score']
        .quantile([0.25, 0.75]).unstack()
        .rename(columns={0.25: 'q1', 0.75: 'q3'}).reset_index()
    )
    counts = (
        finite_train.groupby('location', observed=True)['anomaly_score']
        .size().rename('n_train_scores').reset_index()
    )
    quantile_stats = grouped_quantiles.merge(counts, on='location', validate='one_to_one')
    threshold_formula_mode = 'saved_threshold_anchored_export_iqr'

quantile_stats['location'] = quantile_stats['location'].astype(str)
quantile_stats['iqr'] = quantile_stats['q3'] - quantile_stats['q1']
if (quantile_stats['iqr'] <= 0).any():
    bad = quantile_stats.loc[quantile_stats['iqr'] <= 0, 'location'].tolist()[:10]
    raise ValueError(f'IQR non positivo per le località: {bad}')
threshold_model = quantile_stats.merge(
    saved_thresholds.reset_index(), on='location', how='inner', validate='one_to_one'
)
if len(threshold_model) != len(saved_thresholds):
    raise ValueError('Quantili mancanti per almeno una località presente nel test.')
threshold_model['paper_threshold_k_1_5'] = (
    threshold_model['q3'] + PAPER_IQR_K * threshold_model['iqr']
)
threshold_model['baseline_difference'] = (
    threshold_model['paper_threshold_k_1_5'] - threshold_model['saved_threshold']
)
if TRAIN_QUANTILE_SOURCE == 'full_per_site':
    max_difference = threshold_model['baseline_difference'].abs().max()
    print(f'Massima differenza threshold ricostruita/salvata: {max_difference:.8g}')
    if not np.allclose(
        threshold_model['paper_threshold_k_1_5'],
        threshold_model['saved_threshold'], rtol=1e-6, atol=1e-5,
    ):
        raise ValueError('k=1.5 non ricostruisce le threshold salvate: controllare i file training.')
else:
    print('ATTENZIONE: sweep ancorato al baseline; IQR stimato dagli anni esportati.')
threshold_model.to_csv(OUT_DIR / 'training_iqr_by_location.csv', index=False)

model_by_location = threshold_model.set_index('location')
iqr = joined['location'].map(model_by_location['iqr']).to_numpy(float)
q3 = joined['location'].map(model_by_location['q3']).to_numpy(float)
saved = joined['location'].map(model_by_location['saved_threshold']).to_numpy(float)
score = joined['anomaly_score'].to_numpy(float)
if not np.isfinite(np.column_stack([iqr, q3, saved])).all():
    raise ValueError('Parametri IQR non allineati ad almeno una riga SDE/MTGFlow.')
if TRAIN_QUANTILE_SOURCE == 'full_per_site':
    coordinate = (score - q3) / iqr
else:
    coordinate = PAPER_IQR_K + (score - saved) / iqr
baseline_mismatches = int(np.count_nonzero(
    (coordinate >= PAPER_IQR_K) != (score >= saved)
))
print(f'Decisioni discordanti a k=1.5: {baseline_mismatches}')
if baseline_mismatches:
    raise ValueError('Lo sweep k=1.5 non riproduce la classificazione MTGFlow salvata.')
sweep_values = np.asarray(IQR_K_VALUES, dtype=float)
x_label = 'Threshold MTGFlow: coefficiente IQR k'
baseline_coordinate = PAPER_IQR_K

finite = np.isfinite(coordinate)
coordinate = coordinate[finite]
abs_error = joined['abs_error'].to_numpy(float)[finite]
squared_error = joined['squared_error'].to_numpy(float)[finite]
order = np.argsort(coordinate, kind='stable')
sorted_coordinate = coordinate[order]
sorted_abs = abs_error[order]
sorted_squared = squared_error[order]
prefix_abs = np.concatenate(([0.0], np.cumsum(sorted_abs, dtype=np.float64)))
prefix_squared = np.concatenate(([0.0], np.cumsum(sorted_squared, dtype=np.float64)))

In [ ]:
def threshold_sweep(sorted_values, prefix_absolute, prefix_square, thresholds):
    total_n = len(sorted_values)
    total_abs = prefix_absolute[-1]
    total_square = prefix_square[-1]
    rows = []
    for threshold_value in thresholds:
        # normale: coordinate < threshold; raro: coordinate >= threshold
        split = int(np.searchsorted(sorted_values, threshold_value, side='left'))
        n_normal = split
        n_rare = total_n - split
        normal_abs = prefix_absolute[split]
        normal_square = prefix_square[split]
        rare_abs = total_abs - normal_abs
        rare_square = total_square - normal_square
        mae_normal = normal_abs / n_normal if n_normal else np.nan
        mae_rare = rare_abs / n_rare if n_rare else np.nan
        rmse_normal = np.sqrt(normal_square / n_normal) if n_normal else np.nan
        rmse_rare = np.sqrt(rare_square / n_rare) if n_rare else np.nan
        rows.append({
            'iqr_k': float(threshold_value),
            'n_normal': n_normal, 'n_rare': n_rare,
            'rare_fraction': n_rare / total_n if total_n else np.nan,
            'mae_normal': mae_normal, 'mae_rare': mae_rare,
            'rmse_normal': rmse_normal, 'rmse_rare': rmse_rare,
            'mae_gap': mae_rare - mae_normal,
            'rmse_gap': rmse_rare - rmse_normal,
            'mae_ratio': mae_rare / mae_normal if mae_normal else np.nan,
            'rmse_ratio': rmse_rare / rmse_normal if rmse_normal else np.nan,
        })
    return pd.DataFrame(rows)

sweep = threshold_sweep(sorted_coordinate, prefix_abs, prefix_squared, sweep_values)
effective_rows = []
for k in sweep_values:
    if TRAIN_QUANTILE_SOURCE == 'full_per_site':
        effective = threshold_model['q3'] + k * threshold_model['iqr']
    else:
        effective = (
            threshold_model['saved_threshold']
            + (k - PAPER_IQR_K) * threshold_model['iqr']
        )
    effective_rows.append({
        'iqr_k': float(k),
        'effective_threshold_p25': float(effective.quantile(0.25)),
        'effective_threshold_median': float(effective.median()),
        'effective_threshold_p75': float(effective.quantile(0.75)),
    })
sweep = sweep.merge(pd.DataFrame(effective_rows), on='iqr_k', validate='one_to_one')
sweep['threshold_formula_mode'] = threshold_formula_mode
sweep['daytime_only'] = DAYTIME_ONLY
sweep.to_csv(OUT_DIR / 'threshold_sweep_metrics.csv', index=False)
display(sweep.head())
display(sweep.iloc[[0, len(sweep)//2, len(sweep)-1]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sweep['iqr_k'], sweep['mae_normal'], label='Normali')
axes[0].plot(sweep['iqr_k'], sweep['mae_rare'], label='Rari/anomali')
axes[0].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7, label='Threshold attuale')
axes[0].set(xlabel=x_label, ylabel='MAE [W]', title='MAE in funzione della threshold')
axes[0].grid(alpha=0.25); axes[0].legend()

axes[1].plot(sweep['iqr_k'], sweep['rmse_normal'], label='Normali')
axes[1].plot(sweep['iqr_k'], sweep['rmse_rare'], label='Rari/anomali')
axes[1].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7, label='Threshold attuale')
axes[1].set(xlabel=x_label, ylabel='RMSE [W]', title='RMSE in funzione della threshold')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'error_vs_mtgflow_threshold.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sweep['iqr_k'], 100 * sweep['rare_fraction'], color='tab:red')
axes[0].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7)
axes[0].set(xlabel=x_label, ylabel='Campioni rari [%]', title='Variazione della classificazione')
axes[0].grid(alpha=0.25)
axes[1].plot(sweep['iqr_k'], sweep['n_normal'], label='Normali')
axes[1].plot(sweep['iqr_k'], sweep['n_rare'], label='Rari/anomali')
axes[1].axvline(baseline_coordinate, color='black', linestyle='--', alpha=0.7)
axes[1].set(xlabel=x_label, ylabel='Numero campioni', title='Dimensione dei due gruppi')
axes[1].grid(alpha=0.25); axes[1].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'classification_vs_mtgflow_threshold.png', dpi=180, bbox_inches='tight')
plt.show()

## Scelta della threshold

Non scegliere automaticamente il punto con MAE/RMSE raro massimo: aumentando la threshold il gruppo raro diventa piccolo e instabile. Valutare insieme separazione dell'errore, percentuale rara e numerosità.

`k=1.5` è la scelta a priori del paper MTGFlow. Una scelta differente basata sulle curve del 2019 è un'analisi esplorativa e non una validazione indipendente. Per una threshold finale senza leakage, ripetere la selezione su un anno di validation e congelare `SELECTED_IQR_K` prima della valutazione 2019.

In [ ]:
support = sweep[(sweep['rare_fraction'] >= 0.005) & (sweep['rare_fraction'] <= 0.20)].copy()
print('Candidati con frazione rara tra 0.5% e 20%:')
display(support[[
    'iqr_k', 'rare_fraction', 'n_rare',
    'mae_normal', 'mae_rare', 'mae_ratio',
    'rmse_normal', 'rmse_rare', 'rmse_ratio',
]])
selected_row = sweep.iloc[(sweep['iqr_k'] - SELECTED_IQR_K).abs().argmin()]
print('Threshold selezionata:')
display(selected_row.to_frame('value'))

In [ ]:
location_thresholds = threshold_model[[
    'location', 'q1', 'q3', 'iqr', 'n_train_scores',
    'saved_threshold', 'paper_threshold_k_1_5', 'baseline_difference',
]].copy()
location_thresholds['selected_iqr_k'] = float(SELECTED_IQR_K)
if TRAIN_QUANTILE_SOURCE == 'full_per_site':
    location_thresholds['selected_threshold'] = (
        location_thresholds['q3'] + SELECTED_IQR_K * location_thresholds['iqr']
    )
else:
    location_thresholds['selected_threshold'] = (
        location_thresholds['saved_threshold']
        + (SELECTED_IQR_K - PAPER_IQR_K) * location_thresholds['iqr']
    )
location_thresholds.to_csv(OUT_DIR / 'selected_thresholds_by_location.csv', index=False)
summary = {
    'post_processing_only': True,
    'mtgflow_training_rerun': False,
    'sdenet_training_rerun': False,
    'threshold_formula_mode': threshold_formula_mode,
    'train_quantile_source': TRAIN_QUANTILE_SOURCE,
    'paper_iqr_k': float(PAPER_IQR_K),
    'selected_iqr_k': float(SELECTED_IQR_K),
    'daytime_only': DAYTIME_ONLY,
    'join_coverage': float(coverage),
    'n_analyzed': int(len(joined)),
    'selection_warning': 'Use a validation year for unbiased final threshold selection.',
}
(OUT_DIR / 'threshold_analysis_metadata.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)
print('File prodotti:')
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)